# レッスン7：最終プロジェクト — 既存曲の再現

## このレッスンの目標

好きな曲を1曲選び、レッスン1〜6で学んだ技術を総動員してプログラムで再現します。

### 評価のポイント
- **技術要素の活用度**: 波形生成、エンベロープ、エフェクト、シーケンサーなどをどれだけ活用できたか
- **原曲の再現度**: メロディ、リズム、コード進行がどれだけ正確か
- **コードの品質**: 読みやすく整理されたコードになっているか

### レッスンの流れ
1. まず2つの例題で「既存曲の再現」の手順を学ぶ
2. 楽譜からコードへの変換方法を確認する
3. 音色設計のリファレンスを参照する
4. 自分で選んだ曲を実装する

## 7.1 環境セットアップ

In [ ]:
# 環境検出とセットアップ
import sys
import os

try:
    import google.colab
    IN_COLAB = True
    print("🔧 Google Colab環境で実行中...")
except ImportError:
    IN_COLAB = False
    print("🏠 ローカル環境で実行中")

if IN_COLAB:
    print("🔧 Google Colab環境を設定中...")
    !pip install numpy scipy matplotlib ipython japanize-matplotlib
    !git clone https://github.com/ggszk/simple-audio-programming.git
    sys.path.append('/content/simple-audio-programming')
    print("✅ セットアップ完了！")
else:
    print(f"Python実行ファイル: {sys.executable}")
    print(f"作業ディレクトリ: {os.getcwd()}")
    try:
        import audio_lib
        print("✅ audio_lib ライブラリが利用可能です")
    except ImportError as e:
        raise ImportError("環境が正しくセットアップされていません") from e

print("\n📦 必要なモジュールをimport中...")
from audio_lib import (
    sine_wave, square_wave, sawtooth_wave, triangle_wave, adsr, save_audio, AudioSignal,
    note_to_frequency, note_name_to_number, number_to_note_name,
    Sequencer, Note, Track, create_simple_melody, create_chord,
    BasicPiano, BasicOrgan, BasicGuitar, BasicDrum,
    LowPassFilter, HighPassFilter,
)
from audio_lib.effects.audio_effects import Reverb, Delay, Chorus
from IPython.display import Audio, display
import numpy as np
import matplotlib.pyplot as plt

if IN_COLAB:
    import japanize_matplotlib

import warnings
warnings.filterwarnings('ignore')

sample_rate = 44100
print("✅ セットアップ完了！")

---
## 7.2 例題1：「きらきら星」をリッチに仕上げる

まずは簡単な曲を題材に、段階的に音を豊かにしていく手順を学びます。

### ステップ1：メロディだけを打ち込む（レッスン5の復習）

In [ ]:
# きらきら星のメロディ（ドドソソララソ ファファミミレレド）
# まずはシンプルにピアノで鳴らす

seq = Sequencer()
melody_track = Track(name="melody")

# メロディの音符を定義
# きらきら星: C C G G A A G - F F E E D D C
melody_notes = [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,  # きらきらひかる
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,  # おそらのほしよ
]

create_simple_melody(melody_track, melody_notes, note_duration=0.5)

seq.add_track(melody_track)
seq.set_instrument("melody", BasicPiano())

audio = seq.render()
print("ステップ1：メロディだけ（ピアノ）")
display(Audio(audio.data, rate=sample_rate))

### ステップ2：コード伴奏を追加する

きらきら星のコード進行は `C - F - C - G - C` が基本です。各小節（2拍）に対応するコードを付けます。

In [ ]:
seq = Sequencer()

# メロディトラック（同じ）
melody_track = Track(name="melody")
melody_notes = [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
]
create_simple_melody(melody_track, melody_notes, note_duration=0.5)

# コード伴奏トラック
chord_track = Track(name="chords")

# 各コードを1小節（1秒 = 2拍 x 0.5秒）ずつ配置
# 小節1 (0.0s): C C → Cメジャー
create_chord(chord_track, ["C3", "E3", "G3"], start_time=0.0, duration=1.0)
# 小節2 (1.0s): G G → Cメジャー
create_chord(chord_track, ["C3", "E3", "G3"], start_time=1.0, duration=1.0)
# 小節3 (2.0s): A A → Fメジャー
create_chord(chord_track, ["F3", "A3", "C4"], start_time=2.0, duration=1.0)
# 小節4 (3.0s): G - → Cメジャー
create_chord(chord_track, ["C3", "E3", "G3"], start_time=3.0, duration=1.0)
# 小節5 (4.0s): F F → Fメジャー
create_chord(chord_track, ["F3", "A3", "C4"], start_time=4.0, duration=1.0)
# 小節6 (5.0s): E E → Cメジャー
create_chord(chord_track, ["C3", "E3", "G3"], start_time=5.0, duration=1.0)
# 小節7 (6.0s): D D → Gメジャー
create_chord(chord_track, ["G2", "B2", "D3"], start_time=6.0, duration=1.0)
# 小節8 (7.0s): C - → Cメジャー
create_chord(chord_track, ["C3", "E3", "G3"], start_time=7.0, duration=1.0)

seq.add_track(melody_track)
seq.add_track(chord_track)
seq.set_instrument("melody", BasicPiano())
seq.set_instrument("chords", BasicOrgan())

audio = seq.render()
print("ステップ2：メロディ + コード伴奏")
display(Audio(audio.data, rate=sample_rate))

### ステップ3：ベースラインを追加する

各コードのルート音をベースとして追加します。

In [ ]:
seq = Sequencer()

# メロディトラック
melody_track = Track(name="melody")
melody_notes = [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
]
create_simple_melody(melody_track, melody_notes, note_duration=0.5)

# コードトラック
chord_track = Track(name="chords")
create_chord(chord_track, ["C3", "E3", "G3"], start_time=0.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=1.0, duration=1.0)
create_chord(chord_track, ["F3", "A3", "C4"], start_time=2.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=3.0, duration=1.0)
create_chord(chord_track, ["F3", "A3", "C4"], start_time=4.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=5.0, duration=1.0)
create_chord(chord_track, ["G2", "B2", "D3"], start_time=6.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=7.0, duration=1.0)

# ベーストラック（各コードのルート音）
bass_track = Track(name="bass")
bass_notes = [
    ("C2", 100, 0.0, 1.0),
    ("C2", 100, 1.0, 1.0),
    ("F2", 100, 2.0, 1.0),
    ("C2", 100, 3.0, 1.0),
    ("F2", 100, 4.0, 1.0),
    ("C2", 100, 5.0, 1.0),
    ("G2", 100, 6.0, 1.0),
    ("C2", 100, 7.0, 1.0),
]
bass_track.add_notes(bass_notes)

seq.add_track(melody_track)
seq.add_track(chord_track)
seq.add_track(bass_track)
seq.set_instrument("melody", BasicPiano())
seq.set_instrument("chords", BasicOrgan())
seq.set_instrument("bass", BasicGuitar())

audio = seq.render()
print("ステップ3：メロディ + コード + ベース")
display(Audio(audio.data, rate=sample_rate))

### ステップ4：ドラムパターンを追加する

BasicDrumでは、MIDIノート番号で打楽器の種類を指定します。
- 36 = キック（バスドラム）
- 38 = スネア
- 42 = ハイハット

In [ ]:
seq = Sequencer()

# メロディトラック
melody_track = Track(name="melody")
melody_notes = [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
]
create_simple_melody(melody_track, melody_notes, note_duration=0.5)

# コードトラック
chord_track = Track(name="chords")
create_chord(chord_track, ["C3", "E3", "G3"], start_time=0.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=1.0, duration=1.0)
create_chord(chord_track, ["F3", "A3", "C4"], start_time=2.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=3.0, duration=1.0)
create_chord(chord_track, ["F3", "A3", "C4"], start_time=4.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=5.0, duration=1.0)
create_chord(chord_track, ["G2", "B2", "D3"], start_time=6.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=7.0, duration=1.0)

# ベーストラック
bass_track = Track(name="bass")
bass_notes = [
    ("C2", 100, 0.0, 1.0), ("C2", 100, 1.0, 1.0),
    ("F2", 100, 2.0, 1.0), ("C2", 100, 3.0, 1.0),
    ("F2", 100, 4.0, 1.0), ("C2", 100, 5.0, 1.0),
    ("G2", 100, 6.0, 1.0), ("C2", 100, 7.0, 1.0),
]
bass_track.add_notes(bass_notes)

# ドラムトラック
drum_track = Track(name="drums")
drum_notes = []
for beat in range(16):  # 16拍（8小節 x 2拍）
    t = beat * 0.5
    # ハイハット: 全拍
    drum_notes.append((42, 80, t, 0.25))
    # キック: 小節の頭（偶数拍）
    if beat % 2 == 0:
        drum_notes.append((36, 100, t, 0.25))
    # スネア: 小節の裏（奇数拍）
    if beat % 2 == 1:
        drum_notes.append((38, 90, t, 0.25))
drum_track.add_notes(drum_notes)

seq.add_track(melody_track)
seq.add_track(chord_track)
seq.add_track(bass_track)
seq.add_track(drum_track)
seq.set_instrument("melody", BasicPiano())
seq.set_instrument("chords", BasicOrgan())
seq.set_instrument("bass", BasicGuitar())
seq.set_instrument("drums", BasicDrum())

audio = seq.render()
print("ステップ4：フルアレンジ（メロディ + コード + ベース + ドラム）")
display(Audio(audio.data, rate=sample_rate))

### ステップ5：エフェクトを適用して仕上げる

最後にリバーブを加えて、空間的な広がりを出します。

In [ ]:
# ステップ4と同じ構成をレンダリング
seq = Sequencer()

melody_track = Track(name="melody")
melody_notes = [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
]
create_simple_melody(melody_track, melody_notes, note_duration=0.5)

chord_track = Track(name="chords")
create_chord(chord_track, ["C3", "E3", "G3"], start_time=0.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=1.0, duration=1.0)
create_chord(chord_track, ["F3", "A3", "C4"], start_time=2.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=3.0, duration=1.0)
create_chord(chord_track, ["F3", "A3", "C4"], start_time=4.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=5.0, duration=1.0)
create_chord(chord_track, ["G2", "B2", "D3"], start_time=6.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=7.0, duration=1.0)

bass_track = Track(name="bass")
bass_notes = [
    ("C2", 100, 0.0, 1.0), ("C2", 100, 1.0, 1.0),
    ("F2", 100, 2.0, 1.0), ("C2", 100, 3.0, 1.0),
    ("F2", 100, 4.0, 1.0), ("C2", 100, 5.0, 1.0),
    ("G2", 100, 6.0, 1.0), ("C2", 100, 7.0, 1.0),
]
bass_track.add_notes(bass_notes)

drum_track = Track(name="drums")
drum_notes = []
for beat in range(16):
    t = beat * 0.5
    drum_notes.append((42, 80, t, 0.25))
    if beat % 2 == 0:
        drum_notes.append((36, 100, t, 0.25))
    if beat % 2 == 1:
        drum_notes.append((38, 90, t, 0.25))
drum_track.add_notes(drum_notes)

seq.add_track(melody_track)
seq.add_track(chord_track)
seq.add_track(bass_track)
seq.add_track(drum_track)
seq.set_instrument("melody", BasicPiano())
seq.set_instrument("chords", BasicOrgan())
seq.set_instrument("bass", BasicGuitar())
seq.set_instrument("drums", BasicDrum())

audio = seq.render()

# リバーブを適用
reverb = Reverb(room_size=0.4, damping=0.5, wet=0.2)
audio_with_reverb = reverb.process(audio)

print("ステップ5：完成版（リバーブ適用）")
display(Audio(audio_with_reverb.data, rate=sample_rate))

# 比較用に波形を表示
fig, axes = plt.subplots(2, 1, figsize=(12, 4))
t = np.arange(len(audio.data)) / sample_rate
axes[0].plot(t, audio.data, linewidth=0.5)
axes[0].set_title("エフェクトなし")
axes[0].set_ylabel("振幅")
t_rev = np.arange(len(audio_with_reverb.data)) / sample_rate
axes[1].plot(t_rev, audio_with_reverb.data, linewidth=0.5)
axes[1].set_title("リバーブ適用後")
axes[1].set_xlabel("時間 (秒)")
axes[1].set_ylabel("振幅")
plt.tight_layout()
plt.show()

---
## 7.3 例題2：「かえるの合唱」を輪唱（カノン）で実装

同じメロディを時間差で重ねる「輪唱（カノン）」は、シーケンサーの能力を試すよい題材です。

### メロディの確認
かえるの合唱: ド レ ミ ファ ミ レ ド ー ミ ファ ソ ラ ソ ファ ミ ー

In [ ]:
# かえるの合唱 — 輪唱（カノン）

# メロディの定義
frog_melody = [
    "C4", "D4", "E4", "F4", "E4", "D4", "C4", None,  # ドレミファミレド（休符）
    "E4", "F4", "G4", "A4", "G4", "F4", "E4", None,  # ミファソラソファミ（休符）
]

note_dur = 0.4  # 1音の長さ
round_delay = note_dur * 8  # 8拍遅れで次の声部が入る

seq = Sequencer()

# 声部1（ピアノ）
voice1 = Track(name="voice1")
create_simple_melody(voice1, frog_melody, note_duration=note_dur, start_time=0.0)

# 声部2（オルガン）— 8拍遅れ
voice2 = Track(name="voice2")
create_simple_melody(voice2, frog_melody, note_duration=note_dur, start_time=round_delay)

# 声部3（ギター）— 16拍遅れ
voice3 = Track(name="voice3")
create_simple_melody(voice3, frog_melody, note_duration=note_dur, start_time=round_delay * 2)

seq.add_track(voice1)
seq.add_track(voice2)
seq.add_track(voice3)
seq.set_instrument("voice1", BasicPiano())
seq.set_instrument("voice2", BasicOrgan())
seq.set_instrument("voice3", BasicGuitar())

# 全声部が終わるまでの時間を計算
total_dur = round_delay * 2 + note_dur * len(frog_melody) + 0.5
audio = seq.render(duration=total_dur)

# 軽くリバーブをかける
reverb = Reverb(room_size=0.3, damping=0.5, wet=0.15)
audio = reverb.process(audio)

print("かえるの合唱 — 3声の輪唱")
print(f"  声部1（ピアノ）: 0.0秒から開始")
print(f"  声部2（オルガン）: {round_delay:.1f}秒から開始")
print(f"  声部3（ギター）: {round_delay * 2:.1f}秒から開始")
display(Audio(audio.data, rate=sample_rate))

---
## 7.4 楽譜からコードへの変換ガイド

既存の曲を実装するには、楽譜を読んでコードに変換する必要があります。

### 音符の長さとプログラム上の秒数

テンポ120 BPM（1分間に120拍）の場合：

| 音符 | 名前 | 拍数 | 秒数 |
|------|------|------|------|
| 𝅝 | 全音符 | 4拍 | 2.0秒 |
| 𝅗𝅥 | 2分音符 | 2拍 | 1.0秒 |
| ♩ | 4分音符 | 1拍 | 0.5秒 |
| ♪ | 8分音符 | 0.5拍 | 0.25秒 |
| ♬ | 16分音符 | 0.25拍 | 0.125秒 |

**計算式**: `秒数 = (60 / BPM) * 拍数`

テンポ120なら `60 / 120 = 0.5秒/拍`

### 休符の扱い

`create_simple_melody` では `None` が休符になります。手動で `add_notes` する場合は、その時間に音符を置かなければ自動的に無音です。

### 音名とノート番号

| ド | レ | ミ | ファ | ソ | ラ | シ |
|----|----|----|----|----|----|----|
| C4 | D4 | E4 | F4 | G4 | A4 | B4 |

- `#`（シャープ）: `"C#4"`, `"F#4"` など
- オクターブ: 数字が大きいほど高い音（C3 < C4 < C5）

### 楽譜が読めない場合

- 「曲名 + 楽譜」「曲名 + ピアノ 簡単」で検索
- ドレミが振ってある初心者向け楽譜を探す
- まずメロディだけ打ち込み、コードは後から付ける

In [ ]:
# テンポから秒数を計算するユーティリティ

def beats_to_seconds(beats, bpm=120):
    """拍数を秒数に変換"""
    return beats * (60.0 / bpm)

# 使用例
bpm = 120
print(f"テンポ: {bpm} BPM")
print(f"  4分音符（1拍）: {beats_to_seconds(1, bpm):.3f}秒")
print(f"  8分音符（0.5拍）: {beats_to_seconds(0.5, bpm):.3f}秒")
print(f"  2分音符（2拍）: {beats_to_seconds(2, bpm):.3f}秒")
print(f"  付点4分音符（1.5拍）: {beats_to_seconds(1.5, bpm):.3f}秒")
print()

# テンポ100の場合
bpm_slow = 100
print(f"テンポ: {bpm_slow} BPM")
print(f"  4分音符（1拍）: {beats_to_seconds(1, bpm_slow):.3f}秒")
print(f"  8分音符（0.5拍）: {beats_to_seconds(0.5, bpm_slow):.3f}秒")

---
## 7.5 音色設計リファレンス

楽器クラスを使う方法と、波形＋エンベロープで自作する方法があります。

### 楽器クラスを使う（簡単）

| 音色 | クラス | 特徴 |
|------|--------|------|
| ピアノ風 | `BasicPiano()` | 正弦波＋短めの減衰 |
| オルガン風 | `BasicOrgan()` | 矩形波＋持続的な音 |
| ギター風 | `BasicGuitar()` | のこぎり波＋プラック的な減衰 |
| ドラム風 | `BasicDrum()` | ノート番号で打楽器を指定（36=キック, 38=スネア, 42=ハイハット） |

### 波形＋エンベロープで自作する（応用）

In [ ]:
# 音色設計のデモンストレーション

freq = note_to_frequency(note_name_to_number("C4"))
dur = 1.5

# 1. ピアノ風: 正弦波 + 短い減衰
piano_sound = sine_wave(freq, dur) * adsr(dur, attack=0.01, decay=0.3, sustain=0.3, release=0.2)
print("ピアノ風: sine + 短い減衰")
display(Audio(piano_sound.data, rate=sample_rate))

# 2. オルガン風: 矩形波 + 持続的エンベロープ
organ_sound = square_wave(freq, dur, amplitude=0.3) * adsr(dur, attack=0.05, decay=0.1, sustain=0.8, release=0.2)
print("オルガン風: square + 持続的エンベロープ")
display(Audio(organ_sound.data, rate=sample_rate))

# 3. ギター風: のこぎり波 + プラック減衰
guitar_sound = sawtooth_wave(freq, dur, amplitude=0.3) * adsr(dur, attack=0.005, decay=0.4, sustain=0.1, release=0.3)
print("ギター風: sawtooth + プラック減衰")
display(Audio(guitar_sound.data, rate=sample_rate))

# 4. 8bit風: 矩形波 + シンプルなエンベロープ
bit8_sound = square_wave(freq, dur, amplitude=0.3) * adsr(dur, attack=0.001, decay=0.05, sustain=0.6, release=0.05)
print("8bit風: square + シンプルエンベロープ")
display(Audio(bit8_sound.data, rate=sample_rate))

# 5. パッド風: 正弦波 + ゆっくりしたアタック/リリース + リバーブ
pad_sound = sine_wave(freq, dur) * adsr(dur, attack=0.5, decay=0.2, sustain=0.7, release=0.5)
pad_sound = Reverb(room_size=0.7, wet=0.4).process(pad_sound)
print("パッド風: sine + 長いアタック/リリース + リバーブ")
display(Audio(pad_sound.data, rate=sample_rate))

---
## 7.6 自分の曲を選んで実装しよう

ここからが本番です。好きな曲を1曲選び、上の例題と同じ手順で実装してください。

### 曲選びのヒント
- 童謡や唱歌はメロディがシンプルで取り組みやすい
- ゲーム音楽の8bit曲は矩形波と相性がよい
- テンポが遅い曲の方が打ち込みやすい
- まずは8〜16小節程度の短い部分から始める

### ステップ1：曲の情報を記入する

In [ ]:
# TODO: 自分の曲の情報を記入
song_title = "曲名をここに"  # 例: "チューリップ"
song_bpm = 120                # テンポ（BPM）
beat_duration = 60.0 / song_bpm  # 1拍の秒数（自動計算）

print(f"曲: {song_title}")
print(f"テンポ: {song_bpm} BPM")
print(f"1拍 = {beat_duration:.3f}秒")
print(f"1小節（4拍） = {beat_duration * 4:.3f}秒")

### ステップ2：メロディを打ち込む

In [ ]:
seq = Sequencer()

# TODO: メロディを入力
# create_simple_melody を使う場合（全ての音符が同じ長さ）:
melody_track = Track(name="melody")
my_melody = [
    # TODO: 音名を入力（例: "C4", "D4", "E4", ...）
    # 休符は None
]
create_simple_melody(melody_track, my_melody, note_duration=beat_duration)

# 音符の長さが異なる場合は add_notes を使う:
# melody_track.add_notes([
#     ("C4", 100, 0.0, beat_duration),      # 4分音符
#     ("D4", 100, 0.5, beat_duration * 2),   # 2分音符
#     ("E4", 100, 1.5, beat_duration * 0.5), # 8分音符
# ])

seq.add_track(melody_track)
seq.set_instrument("melody", BasicPiano())  # TODO: 楽器を選ぶ

audio = seq.render()
print("メロディの確認")
display(Audio(audio.data, rate=sample_rate))

### ステップ3：コード伴奏を追加する

In [ ]:
# TODO: コード伴奏を追加
chord_track = Track(name="chords")

# 各小節にコードを配置
# create_chord(chord_track, ["C3", "E3", "G3"], start_time=0.0, duration=beat_duration * 4)
# create_chord(chord_track, ["F3", "A3", "C4"], start_time=beat_duration * 4, duration=beat_duration * 4)
# ...

seq.add_track(chord_track)
seq.set_instrument("chords", BasicOrgan())  # TODO: 楽器を選ぶ

audio = seq.render()
print("メロディ + コードの確認")
display(Audio(audio.data, rate=sample_rate))

### ステップ4：ベースラインを追加する

In [ ]:
# TODO: ベースラインを追加（コードのルート音を使うのが基本）
bass_track = Track(name="bass")

# bass_track.add_notes([
#     ("C2", 100, 0.0, beat_duration * 4),
#     ("F2", 100, beat_duration * 4, beat_duration * 4),
#     ...
# ])

seq.add_track(bass_track)
seq.set_instrument("bass", BasicGuitar())  # TODO: 楽器を選ぶ

audio = seq.render()
print("メロディ + コード + ベースの確認")
display(Audio(audio.data, rate=sample_rate))

### ステップ5：ドラムパターンを追加する

In [ ]:
# TODO: ドラムパターンを追加
# ドラム: 36=キック, 38=スネア, 42=ハイハット
drum_track = Track(name="drums")

# 例: 基本的な4拍子パターン
# total_beats = ...  # 曲全体の拍数
# drum_notes = []
# for beat in range(total_beats):
#     t = beat * beat_duration
#     drum_notes.append((42, 80, t, 0.1))    # ハイハット: 全拍
#     if beat % 4 == 0:
#         drum_notes.append((36, 100, t, 0.2))  # キック: 小節頭
#     if beat % 4 == 2:
#         drum_notes.append((38, 90, t, 0.2))   # スネア: 3拍目
# drum_track.add_notes(drum_notes)

seq.add_track(drum_track)
seq.set_instrument("drums", BasicDrum())

audio = seq.render()
print("フルアレンジの確認")
display(Audio(audio.data, rate=sample_rate))

### ステップ6：エフェクトを適用する

In [ ]:
# TODO: エフェクトを適用して仕上げる

# リバーブ
# reverb = Reverb(room_size=0.4, damping=0.5, wet=0.2)
# audio = reverb.process(audio)

# ディレイ
# delay = Delay(delay_time=0.3, feedback=0.2, wet=0.15)
# audio = delay.process(audio)

# ローパスフィルター（高音を抑えたい場合）
# lpf = LowPassFilter(cutoff=3000)
# audio = lpf.process(audio)

print("最終版")
display(Audio(audio.data, rate=sample_rate))

### ステップ7：最終ミックス（まとめて実行）

上のステップで確認しながら作った内容を、1つのセルにまとめて最終版を作りましょう。

In [ ]:
# ===== 最終ミックス =====
# TODO: 上のステップで作った全パートをまとめる

seq = Sequencer()

# --- メロディ ---
melody_track = Track(name="melody")
# TODO: メロディの音符をここに

# --- コード ---
chord_track = Track(name="chords")
# TODO: コード進行をここに

# --- ベース ---
bass_track = Track(name="bass")
# TODO: ベースラインをここに

# --- ドラム ---
drum_track = Track(name="drums")
# TODO: ドラムパターンをここに

# --- トラック登録と楽器設定 ---
seq.add_track(melody_track)
seq.add_track(chord_track)
seq.add_track(bass_track)
seq.add_track(drum_track)
seq.set_instrument("melody", BasicPiano())
seq.set_instrument("chords", BasicOrgan())
seq.set_instrument("bass", BasicGuitar())
seq.set_instrument("drums", BasicDrum())

# --- レンダリング ---
audio = seq.render()

# --- エフェクト ---
# TODO: 必要なエフェクトを適用

# --- 再生 ---
print(f"完成: {song_title}")
display(Audio(audio.data, rate=sample_rate))

---
## 7.7 発表シート

実装が完了したら、以下の項目を記入してください。

### 作品情報

- **選んだ曲**: （曲名を記入）
- **選んだ理由**: （なぜこの曲を選んだか）

### 使用した技術要素

以下のうち、使用したものにチェックを入れてください。

- [ ] 波形生成（sine, square, sawtooth, triangle）
- [ ] ADSRエンベロープ
- [ ] シーケンサーによるマルチトラック構成
- [ ] 楽器クラス（BasicPiano, BasicOrgan, BasicGuitar, BasicDrum）
- [ ] コード（和音）
- [ ] エフェクト（リバーブ、ディレイ、フィルターなど）

### 音色設計で工夫した点

（どの楽器にどの音色を使い、なぜそれを選んだか）

### 原曲との違いと、その理由

（技術的制約による違い、意図的に変更した部分など）

### 実装で苦労した点と解決方法

（つまずいたポイントと、どう解決したか）

---
## 7.8 ボーナスチャレンジ

余力がある人は、以下のチャレンジに挑戦してみましょう。

### チャレンジ1：同じ曲を違うジャンルにアレンジ

自分が実装した曲を、まったく違う雰囲気にアレンジしてみましょう。

例:
- **8bit風**: 全パートを `square_wave` で、エフェクトなし
- **アンビエント風**: `sine_wave` + 長いアタック/リリース + 深いリバーブ
- **ロック風**: `sawtooth_wave` + ドラムを強調

In [ ]:
# チャレンジ1：アレンジ版
# TODO: 同じメロディを違う音色・エフェクトで再実装

### チャレンジ2：パラメータ化して演奏条件を変える

テンポやキー（調）を変数にして、簡単に変更できるようにしてみましょう。

In [ ]:
# チャレンジ2：パラメータ化

def render_song(bpm=120, transpose=0):
    """曲をレンダリングする関数

    Args:
        bpm: テンポ（BPM）
        transpose: 移調量（半音単位、例: +2 で全音上げ）
    """
    bt = 60.0 / bpm  # noqa: F841  # 1拍の秒数

    seq = Sequencer()
    melody_track = Track(name="melody")

    # TODO: 自分の曲のメロディを入れる
    # transposeを適用する例:
    # original_notes = ["C4", "D4", "E4", "F4"]
    # for i, note_name in enumerate(original_notes):
    #     midi_num = note_name_to_number(note_name) + transpose
    #     melody_track.add_note(midi_num, 100, i * bt, bt)

    seq.add_track(melody_track)
    seq.set_instrument("melody", BasicPiano())
    return seq.render()

# 通常版
# print("通常 (120 BPM, 原調)")
# display(Audio(render_song(bpm=120, transpose=0).data, rate=sample_rate))

# テンポ変更
# print("スロー (80 BPM)")
# display(Audio(render_song(bpm=80, transpose=0).data, rate=sample_rate))

# キー変更（長3度上げ）
# print("キー +4 (長3度上)")
# display(Audio(render_song(bpm=120, transpose=4).data, rate=sample_rate))

---
## 7.9 コース振り返り

全7回のレッスンで学んだことを振り返りましょう。

### 学んだ技術の積み上げ

| レッスン | テーマ | 学んだこと |
|----------|--------|------------|
| 01 | 音の基礎と正弦波 | 周波数・振幅・位相、正弦波の生成 |
| 02 | エンベロープとADSR | 音の時間変化、ADSRパラメータ |
| 03 | フィルターと音色設計 | ローパス/ハイパスフィルター、倍音構造 |
| 04 | エフェクトとダイナミクス | リバーブ、ディレイ、コーラス |
| 05 | MIDIとシーケンサー | ノート、トラック、マルチパート |
| 06 | サンプリングと分析 | サンプリング定理、FFT、スペクトル分析 |
| 07 | 最終プロジェクト | 全技術を統合して1曲を再現 |

### このコースで身についたこと

- **音の物理**: 音が波であること、周波数が音の高さを決めること
- **信号処理の基礎**: デジタル音声がどのように生成・加工されるか
- **プログラミングによる実装力**: 理論を実際のコードに落とし込む力

### 次のステップ

このコースの内容をさらに深めたい場合は、以下の方向に進むことができます。

- **信号処理をさらに学ぶ**: FIR/IIRフィルターの設計、窓関数、周波数領域での処理
- **シンセサイザーの仕組み**: FM合成、加算合成、減算合成、ウェーブテーブル合成
- **音声プログラミングツール**: SuperCollider, Pure Data, Max/MSP
- **Python音声ライブラリ**: librosa（音声分析）, pydub（音声編集）, pedalboard（エフェクト）

---

お疲れさまでした！全レッスン完了です。

このコースでは「音とは何か」という物理の話から始まり、信号処理の基礎を学び、最終的にプログラムで1曲を再現するところまで到達しました。ここで学んだ知識は、音楽制作ソフトの仕組みを理解したり、音声処理のアプリケーションを開発したりする際の土台になります。